# Assessing the Security of LLM-Generated Code in Programming Tasks
## Statistical Analysis Notebook
### Install dependencies using `pip install -r requirements.txt`

**Study:** Does security-aware prompting improve the security of LLM-generated Python code without substantially reducing functional correctness?

**Models evaluated:** GPT-4o · Claude 3.5 Sonnet · Gemini 1.5 Pro  
**Tasks:** 15 tasks across 5 vulnerability-prone categories  
**Metrics:** Vulnerability Density · Functional Correctness (Pass Rate)  
**Statistical test:** Wilcoxon Signed-Rank Test (α = 0.05)


## 0. Imports & Data Loading

In [17]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

FILE = '../results/aggregate/20260408_174011/combined_scores_summary.xlsx'
by_task  = pd.read_excel(FILE, sheet_name='by_task')
by_model = pd.read_excel(FILE, sheet_name='by_mode_model')
by_mode  = pd.read_excel(FILE, sheet_name='by_mode')

print("Sheets loaded successfully.")
print(f"  by_task  : {by_task.shape[0]} rows | columns: {by_task.columns.tolist()}")
print(f"  by_model : {by_model.shape[0]} rows")
print(f"  by_mode  : {by_mode.shape[0]} rows")
print(f"\nUnique tasks  : {by_task['task'].nunique()}")
print(f"Models        : {by_task['model'].unique().tolist()}")
print(f"Modes         : {by_task['mode'].unique().tolist()}")


Sheets loaded successfully.
  by_task  : 90 rows | columns: ['task', 'mode', 'model', 'pass_rate', 'security_findings', 'vulnerability_density', 'security_score', 'overall_score', 'functional_success', 'secure_success']
  by_model : 6 rows
  by_mode  : 2 rows

Unique tasks  : 15
Models        : ['claude', 'gemini', 'gpt']
Modes         : ['baseline', 'security_aware']


## 1. Vulnerability Density Reduction

**Formula:**

$$\text{VD Reduction (\%)} = \frac{\text{VD}_{baseline} - \text{VD}_{secure}}{\text{VD}_{baseline}} \times 100$$

Where **Vulnerability Density** = (number of security findings / lines of code) × 100


In [18]:
baseline_vd = by_mode.loc[by_mode['mode'] == 'baseline',       'avg_vulnerability_density'].values[0]
secure_vd   = by_mode.loc[by_mode['mode'] == 'security_aware', 'avg_vulnerability_density'].values[0]

vd_reduction = ((baseline_vd - secure_vd) / baseline_vd) * 100

print("=== Vulnerability Density Reduction ===")
print(f"  Baseline avg VD        : {baseline_vd:.4f}")
print(f"  Security-aware avg VD  : {secure_vd:.4f}")
print(f"  Difference             : {baseline_vd:.4f} - {secure_vd:.4f} = {baseline_vd - secure_vd:.4f}")
print(f"  Reduction              : ({baseline_vd:.4f} - {secure_vd:.4f}) / {baseline_vd:.4f} x 100")
print(f"                         = {vd_reduction:.1f}%")

print("\n--- Per Model ---")
for _, row in by_model.iterrows():
    print(f"  {row['mode']:15s} | {row['model']:6s} | avg VD = {row['avg_vulnerability_density']:.2f}")


=== Vulnerability Density Reduction ===
  Baseline avg VD        : 17.5963
  Security-aware avg VD  : 10.4483
  Difference             : 17.5963 - 10.4483 = 7.1480
  Reduction              : (17.5963 - 10.4483) / 17.5963 x 100
                         = 40.6%

--- Per Model ---
  baseline        | claude | avg VD = 28.40
  baseline        | gpt    | avg VD = 12.31
  baseline        | gemini | avg VD = 12.08
  security_aware  | gpt    | avg VD = 8.19
  security_aware  | gemini | avg VD = 9.56
  security_aware  | claude | avg VD = 13.59


## 2. Functional Correctness (Pass Rate)

**Formula:**

$$\text{Change (pp)} = \text{PR}_{secure} - \text{PR}_{baseline}$$

Where **Pass Rate** = percentage of pytest test cases passed per task.


In [19]:
baseline_pr = by_mode.loc[by_mode['mode'] == 'baseline',       'avg_pass_rate'].values[0]
secure_pr   = by_mode.loc[by_mode['mode'] == 'security_aware', 'avg_pass_rate'].values[0]

pr_change = secure_pr - baseline_pr

print("=== Functional Correctness (Pass Rate) ===")
print(f"  Baseline avg pass rate       : {baseline_pr:.2f}%")
print(f"  Security-aware avg pass rate : {secure_pr:.2f}%")
print(f"  Change                       : {secure_pr:.2f} - {baseline_pr:.2f} = {pr_change:+.2f} pp")

print("\n--- Per Model ---")
for _, row in by_model.iterrows():
    print(f"  {row['mode']:15s} | {row['model']:6s} | avg PR = {row['avg_pass_rate']:.1f}%")


=== Functional Correctness (Pass Rate) ===
  Baseline avg pass rate       : 39.04%
  Security-aware avg pass rate : 57.77%
  Change                       : 57.77 - 39.04 = +18.73 pp

--- Per Model ---
  baseline        | claude | avg PR = 40.1%
  baseline        | gpt    | avg PR = 40.8%
  baseline        | gemini | avg PR = 36.2%
  security_aware  | gpt    | avg PR = 57.9%
  security_aware  | gemini | avg PR = 57.8%
  security_aware  | claude | avg PR = 57.7%


## 3. Wilcoxon Signed-Rank Test

**Why Wilcoxon?**  
Each task appears under both conditions (baseline and security-aware) — making this a **paired** design. The Wilcoxon signed-rank test is appropriate because:
- Data is paired (same task, same model, two conditions)
- We cannot assume normality with only n=15 pairs
- It tests whether the median difference between pairs is zero

**H₀:** No difference in the metric between baseline and security-aware prompting  
**H₁:** A difference exists (two-sided)  
**α = 0.05**

> **Note:** Tests are run per model (n=15 pairs each). Pooling all models into n=45 would inflate the sample size by treating the same task repeated across models as independent observations.


In [12]:
models  = ['claude', 'gemini', 'gpt']
metrics = {
    'vulnerability_density' : 'Vulnerability Density',
    'pass_rate'             : 'Pass Rate (Functional Correctness)'
}

results = []

for metric_col, metric_name in metrics.items():
    print(f"{'='*60}")
    print(f"METRIC: {metric_name}")
    print(f"{'='*60}")

    for model in models:
        base = by_task[(by_task['model'] == model) & (by_task['mode'] == 'baseline')].sort_values('task')
        sec  = by_task[(by_task['model'] == model) & (by_task['mode'] == 'security_aware')].sort_values('task')

        merged = base[['task', metric_col]].merge(
            sec[['task', metric_col]], on='task', suffixes=('_base', '_sec')
        )

        b_vals = merged[f'{metric_col}_base'].values
        s_vals = merged[f'{metric_col}_sec'].values
        diffs  = b_vals - s_vals

        if np.all(diffs == 0):
            print(f"  [{model.upper()}] All differences zero — test not applicable.")
            continue

        stat, p  = stats.wilcoxon(b_vals, s_vals, alternative='two-sided')
        n        = len(diffs)
        median_d = np.median(diffs)
        sig      = "SIGNIFICANT ***" if p < 0.05 else "not significant"

        print(f"\n  [{model.upper()}]")
        print(f"    n pairs           : {n}")
        print(f"    Median difference : {median_d:.4f}  (baseline minus secure)")
        print(f"    W statistic       : {stat:.1f}")
        print(f"    p-value           : {p:.4f}  ->  {sig}")

        results.append({
            'Metric'      : metric_name,
            'Model'       : model,
            'n'           : n,
            'Median Diff' : round(median_d, 4),
            'W'           : stat,
            'p-value'     : round(p, 4),
            'Significant' : p < 0.05
        })

    print()

results_df = pd.DataFrame(results)


METRIC: Vulnerability Density

  [CLAUDE]
    n pairs           : 15
    Median difference : 11.5460  (baseline minus secure)
    W statistic       : 14.0
    p-value           : 0.0157  ->  SIGNIFICANT ***

  [GEMINI]
    n pairs           : 15
    Median difference : 1.2228  (baseline minus secure)
    W statistic       : 30.0
    p-value           : 0.1578  ->  not significant

  [GPT]
    n pairs           : 15
    Median difference : 3.1408  (baseline minus secure)
    W statistic       : 12.0
    p-value           : 0.0192  ->  SIGNIFICANT ***

METRIC: Pass Rate (Functional Correctness)

  [CLAUDE]
    n pairs           : 15
    Median difference : -12.5000  (baseline minus secure)
    W statistic       : 9.0
    p-value           : 0.1097  ->  not significant

  [GEMINI]
    n pairs           : 15
    Median difference : -33.3000  (baseline minus secure)
    W statistic       : 28.0
    p-value           : 0.1237  ->  not significant

  [GPT]
    n pairs           : 15
    Media

## 4. Summary Results Table

In [13]:
print("=== WILCOXON SIGNED-RANK TEST SUMMARY ===\n")
print(results_df.to_string(index=False))

print("\n=== INTERPRETATION ===")
print("  Vulnerability Density : positive median diff = baseline had MORE vulnerabilities")
print("                          security-aware prompting REDUCED vulnerability density")
print("  Pass Rate             : negative median diff = baseline had LOWER pass rate")
print("                          security-aware prompting IMPROVED functional correctness")


=== WILCOXON SIGNED-RANK TEST SUMMARY ===

                            Metric  Model  n  Median Diff    W  p-value  Significant
             Vulnerability Density claude 15      11.5460 14.0   0.0157         True
             Vulnerability Density gemini 15       1.2228 30.0   0.1578        False
             Vulnerability Density    gpt 15       3.1408 12.0   0.0192         True
Pass Rate (Functional Correctness) claude 15     -12.5000  9.0   0.1097        False
Pass Rate (Functional Correctness) gemini 15     -33.3000 28.0   0.1237        False
Pass Rate (Functional Correctness)    gpt 15     -16.7000  6.5   0.0322         True

=== INTERPRETATION ===
  Vulnerability Density : positive median diff = baseline had MORE vulnerabilities
                          security-aware prompting REDUCED vulnerability density
  Pass Rate             : negative median diff = baseline had LOWER pass rate
                          security-aware prompting IMPROVED functional correctness


## 5. Key Numbers for the Abstract

In [20]:
print("=== KEY NUMBERS FOR ABSTRACT ===\n")
print(f"1. Vulnerability Density Reduction  : {vd_reduction:.1f}%")
print(f"   from {baseline_vd:.2f} to {secure_vd:.2f} avg VD\n")
print(f"2. Functional Correctness Change    : {pr_change:+.1f} percentage points")
print(f"   from {baseline_pr:.1f}% to {secure_pr:.1f}% avg pass rate\n")

print("3. Wilcoxon Signed-Rank Tests (per model, n=15 pairs each):\n")

vd_res = results_df[results_df['Metric'] == 'Vulnerability Density']
pr_res = results_df[results_df['Metric'] == 'Pass Rate (Functional Correctness)']

print("   Vulnerability Density:")
for _, r in vd_res.iterrows():
    tag = f"p={r['p-value']:.3f} ***" if r['Significant'] else f"p={r['p-value']:.3f} (ns)"
    print(f"     {r['Model']:6s} : W={r['W']:.0f}, {tag}")

print("\n   Pass Rate:")
for _, r in pr_res.iterrows():
    tag = f"p={r['p-value']:.3f} ***" if r['Significant'] else f"p={r['p-value']:.3f} (ns)"
    print(f"     {r['Model']:6s} : W={r['W']:.0f}, {tag}")

abstract_sentence = (
    "Results show that security-aware prompting reduces vulnerability density by "
    f"{vd_reduction:.1f}% overall, with statistically significant improvements for "
    "Claude (W=14, p=0.016) and GPT-4o (W=12, p=0.019) but not Gemini (W=30, p=0.158). "
    f"Functional correctness improved by {pr_change:.1f} percentage points overall, "
    "reaching significance for GPT-4o (W=6.5, p=0.032)."
)
print("\n=== UPDATED ABSTRACT SENTENCE ===\n")
print(abstract_sentence)


=== KEY NUMBERS FOR ABSTRACT ===

1. Vulnerability Density Reduction  : 40.6%
   from 17.60 to 10.45 avg VD

2. Functional Correctness Change    : +18.7 percentage points
   from 39.0% to 57.8% avg pass rate

3. Wilcoxon Signed-Rank Tests (per model, n=15 pairs each):

   Vulnerability Density:
     claude : W=14, p=0.016 ***
     gemini : W=30, p=0.158 (ns)
     gpt    : W=12, p=0.019 ***

   Pass Rate:
     claude : W=9, p=0.110 (ns)
     gemini : W=28, p=0.124 (ns)
     gpt    : W=6, p=0.032 ***

=== UPDATED ABSTRACT SENTENCE ===

Results show that security-aware prompting reduces vulnerability density by 40.6% overall, with statistically significant improvements for Claude (W=14, p=0.016) and GPT-4o (W=12, p=0.019) but not Gemini (W=30, p=0.158). Functional correctness improved by 18.7 percentage points overall, reaching significance for GPT-4o (W=6.5, p=0.032).
